# RhymeLM v2 — LSTM Training Notebook

Train the character-level LSTM model on a dual corpus of English dictionary words
(vocabulary grounding) and rap lyrics (style, structure, rhyme schemes).

This notebook uses the `rhymelm` package — run `pip install -e .` from the project root first.

In [ ]:
from functools import partial

from rhymelm.config import Config
from rhymelm.data import load_dictionary, load_lyrics, extract_verses, build_dual_corpus
from rhymelm.data import CharTokenizer, create_dataloaders
from rhymelm.models import RhymeLM
from rhymelm.training.trainer import train
from rhymelm.generation.sampler import generate_verse
from rhymelm.utils import get_device, seed_everything, count_parameters

## 1. Configuration

In [ ]:
config = Config.load("../configs/lstm_base.yaml")
config.data.csv_path = "../lyrics_raw.csv"

seed_everything(config.training.seed)
device = get_device()
print(f"Device: {device}")

## 2. Data Pipeline

Build the dual corpus: interleave English dictionary words with 16-bar verse chunks.
The dictionary teaches the model what words look like; the lyrics teach it how artists flow.

In [ ]:
dictionary = load_dictionary()
lyrics = load_lyrics(config.data.csv_path, config.data.lyrics_column)
verses = extract_verses(lyrics, config.data.min_bars, config.data.max_bars)

import random
print("\nSample verse:")
print("=" * 50)
print(random.choice(verses))
print("=" * 50)

In [ ]:
corpus = build_dual_corpus(
    verses, dictionary,
    dict_ratio=config.data.dict_ratio,
    seed=config.training.seed,
)

tokenizer = CharTokenizer.from_corpus(corpus)
encoded = tokenizer.encode(corpus)

train_loader, val_loader = create_dataloaders(
    encoded,
    block_size=config.training.block_size,
    batch_size=config.training.batch_size,
    val_split=config.data.val_split,
)

## 3. Model

LSTM backbone: character embedding (256-dim) → 2-layer LSTM (512 hidden) → linear projection.

In [ ]:
model = RhymeLM(
    vocab_size=tokenizer.vocab_size,
    embed_dim=config.model.embed_dim,
    hidden_dim=config.model.hidden_dim,
    num_layers=config.model.num_layers,
    dropout=config.model.dropout,
).to(device)

print(f"Parameters: {count_parameters(model):,}")
print(model)

## 4. Train

Training features:
- **Gradient accumulation** for effective batch size scaling
- **Mixed precision (AMP)** when GPU supports it
- **Linear warmup + cosine decay** LR schedule
- **Decoupled weight decay** (only on weight matrices, not biases/norms)
- **Gradient clipping** at max_norm=1.0

In [ ]:
gen_fn = partial(
    generate_verse,
    model=model,
    tokenizer=tokenizer,
    device=device,
    num_bars=8,
    temperature=0.7,
)

history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,
    config=config,
    device=device,
    generate_fn=gen_fn,
)

## 5. Training Visualization

In [ ]:
from rhymelm.visualization.training_plots import plot_training_dashboard

if history["steps"]:
    plot_training_dashboard(
        steps=history["steps"],
        train_losses=history["train_loss"],
        val_losses=history["val_loss"],
        learning_rates=history["learning_rates"],
        grad_norms=history["grad_norms"],
    )

## 6. Generate Verses

In [ ]:
print("Temperature 0.7:")
print(generate_verse(model, tokenizer, device, prompt="Yeah, ", temperature=0.7, num_bars=16))
print("\n" + "=" * 50 + "\n")
print("Temperature 0.9 + top-p 0.95:")
print(generate_verse(model, tokenizer, device, prompt="I ", temperature=0.9, top_p=0.95, num_bars=16))

## 7. Evaluate

In [ ]:
from rhymelm.evaluation.metrics import evaluate_verse

verse = generate_verse(model, tokenizer, device, temperature=0.8, num_bars=16)
print(verse)
print("\nMetrics:")
metrics = evaluate_verse(verse)
for k, v in metrics.items():
    print(f"  {k}: {v}")